# Arsitektur
Input Layer
Attention Layer
Output Layer

## Data Loader + Preprocessing

In [ ]:
# Load data to dataframe pandas
import pandas as pd

df = pd.read_csv('data_train/1111_2_3_1/mmwave_ss_heart.csv', usecols=["log_time", "heart_rate"])
# usecols=['col1', 'col2'] to select specific columns
# sep='\t' to specify a different delimiter
df.rename(columns={'log_time': 'timestamp', 'heart_rate': 'hr'}, inplace=True)

# Change timestamp string to unix timestamp (no need anymore)
# df['timestamp'] = pd.to_datetime(df['timestamp']).astype(int) / 10**9
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [ ]:
# TODO : Labeling
# Pilihan : Propagasi/Forward Fill; Interpolasi Linier; Borders Exclusion
df["label"] = 0
df.head()

In [ ]:
# Imputation n Resampling
# TODO : Compare between resampling and not resampling
df = df.set_index("timestamp")
df_resampled = df.resample("1s").mean().interpolate(method="linear")

df_resampled = df_resampled.reset_index()
df = df.reset_index()

df_resampled.info()
df.info()

df = df_resampled

In [ ]:
# TODO : Split data into train, val, test
# split data into 80% training 20% test
cutting_point = int(len(df) * 0.8)
trainval_df = df.iloc[:cutting_point]
test_df = df.iloc[cutting_point:]
cutting_point = int(len(trainval_df) * 0.8)
train_df = trainval_df.iloc[:cutting_point]
val_df = trainval_df.iloc[cutting_point:]

print(train_df.shape, test_df.shape, val_df.shape)

In [ ]:
# PREP : Normalization or Standardization
# TODO : Standarization using torch to make it faster
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_df['hr'] = scaler.fit_transform(train_df[['hr']])
val_df['hr'] = scaler.transform(val_df[['hr']])
test_df['hr'] = scaler.transform(test_df[['hr']])

val_df.info()

In [ ]:
# custom dataset for PyTorch
import torch
from torch.utils.data import Dataset
import config

class TimeSeriesDataset(Dataset):
    def __init__(self, data):
        self.features = data[["hr"]].values
        self.labels = data["label"].values

        self.seq_length = config.SEQ_LENGTH
        self.stride = config.WINDOW_STRIDE # offset between sliding windows
        self.context_length = config.CONTEXT_TIME * self.seq_length # 5 minutes
        self.prediction_length = config.PREDICTION_TIME * self.seq_length # 1 minutes
        self.total_window_needed = self.context_length + self.prediction_length # 360 data point

    def __len__(self):
        # total sliding windows that can be made from the data
        return (len(self.features) - self.total_window_needed) // self.stride + 1

    def __getitem__(self, idx):
        # PREP : Sliding Window / context length + prediction length
        # Context Length : Prediction Length = 5 : 1
        x = self.features[idx : idx + self.context_length]
        y_classification = self.labels[idx + self.context_length - 1]
        start_pred_idx = idx + self.context_length # + offset to get the start of prediction window
        end_pred_idx = start_pred_idx + self.prediction_length
        y_forecasting = self.features[start_pred_idx : end_pred_idx]

        # TODO : add label for future classification (use Modus/Max)


        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y_classification, dtype=torch.long),
            torch.tensor(y_forecasting, dtype=torch.float32)
        )

train_dataset = TimeSeriesDataset(train_df)
val_dataset = TimeSeriesDataset(val_df)

In [ ]:
# Batching : decide how much data to load at once during training
from torch.utils.data import DataLoader

batch_size = 32

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    shuffle=True
)

In [ ]:
# Checking before go to model
# 1. Dimensi Tensor must be [batch_size, seq_length, channels]
# 2. Check nilai inf/NaN
for idx, n in enumerate(train_loader):
    print(idx, n[0].shape, n[1].shape, n[2].shape)
    break
print(len(train_loader))

## Model

In [ ]:
# INPUT LAYER
# 1. Embedding 
# 2. Positional Encoding

import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1): # max_len == seq_len
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # shape = [max_length, 1]
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term) # pos * e**(-2i.ln(10000)/d_model)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) # shape = [1, max_len, d_model] so it suitable for batching [batch_size, seq_len, channels]
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :] # if input [Batch, Seq, Channels], use size(1) (size from seq_len)
        return self.dropout(x)


In [ ]:
class TimeSeriesTransformer(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers, output_dim, dropout=0.1):
        super(TimeSeriesTransformer, self).__init__()

        # Input Layer
        self.input_projection = nn.Linear(input_dim, d_model) #input_dim = channels
        self.positional_encoding = PositionalEncoding(d_model, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead,
            dropout=dropout,
            batch_first=True # cuz format [batch_size, seq_len, channels]
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, 
            num_layers=num_layers
        )

        # Output Layer
        self.kss_head = nn.Linear(d_model, 1) # approaching kss score using regression
        self.forecasting_head = nn.Linear(d_model, config.SEQ_LENGTH * config.PREDICTION_TIME)

    def mask_future(self, seq_length):
        mask = torch.triu(torch.ones(seq_length, seq_length) * float('-inf'), diagonal=1)
        return mask

    def forward(self, x):
        x = self.input_projection(x) * math.sqrt(self.input_projection.out_features) # [batch_size, seq_length, d_model] 
        x = self.positional_encoding(x)
        # x = x.permute(1, 0, 2) # Transformer expects [seq_length, batch_size, d_model] (not anymore)

        mask = self.mask_future(x.size(1)).to(x.device) # change it to use is_causal=True
        encoded = self.transformer_encoder(x, mask) # [seq_length, batch_size, d_model]
        # encoded = encoded.permute(1, 0, 2) # back to [batch_size, seq_length, d_model]
        
        cls_token = encoded[:, -1, :] # use the last token for classification
        classification_output = self.kss_head(cls_token)
        forecasting_output = self.forecasting_head(cls_token)

        return classification_output, forecasting_output


In [ ]:
import config

model = TimeSeriesTransformer(
    input_dim=config.INPUT_DIM,
    d_model=config.D_MODEL,
    nhead=config.NUM_HEADS,
    num_layers=config.NUM_LAYERS,
    output_dim=config.OUTPUT_DIM
)

# Training

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training using {device}")

model = model.to(device)

# Define Loss Function
classification_criterion = nn.MSELoss()
forecasting_criterion = nn.MSELoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=config.L_RATE)

In [ ]:
for batch_idx, (data, label, future_data) in enumerate(train_loader):
    pred_label, pred_forecast = model(data)
    print(pred_forecast.shape, future_data.shape)

    break


In [ ]:
import time
import os

history = {'train_loss' : [], 
           'train_kss_mae': [],
           'train_kss_acc': [],
           'train_forecast_mae': [],
           'train_forecast_rmse': [],
           'val_loss' : [],
           'val_kss_mae': [],
           'val_kss_acc': [],
           'val_forecast_mae': [],
           'val_forecast_rmse': []
        }

best_val_loss = float('inf')
timestamp = time.time()

for epoch in range (config.EPOCH):
    # TRAINING
    model.train()
    total_train_loss = 0.0
    train_kss_mae_sum = 0.0
    train_kss_acc_sum = 0.0
    train_forecast_mae_sum = 0.0
    train_forecast_mse_sum = 0.0

    total_samples = 0

    for batch_idx, (data, label, future_data) in enumerate(train_loader):
        # Move data to GPU if available
        data = data.to(device)
        label = label.float().unsqueeze(1).to(device)
        future_data = future_data.to(device)

        batch_size = data.size(0)
        total_samples += batch_size

        # Set gradient from last epoch to zero
        optimizer.zero_grad()

        # get prediction
        pred_kss, pred_forecast = model(data)
        pred_forecast = pred_forecast.unsqueeze(2)

        # calculate loss
        loss_kss = classification_criterion(pred_kss, label)
        loss_forecast = forecasting_criterion(pred_forecast, future_data)
        batch_loss = (config.ALPHA * loss_kss) + ((1 - config.ALPHA) * loss_forecast)
        batch_loss.backward()

        # gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        
        total_train_loss += batch_loss.item()

        # Calculate Metrics
        # 1. KSS MAE
        kss_mae = torch.abs(pred_kss - label).mean().item()
        train_kss_mae_sum += kss_mae * batch_size

        # 2. KSS Accuracy
        rounded_kss = torch.clamp(torch.round(pred_kss), min=1.0, max=9.0)
        correct_kss = (rounded_kss == label).sum().item()
        train_kss_acc_sum += correct_kss

        # 3. Forecast MAE
        forecast_mae = torch.abs(pred_forecast - future_data).mean().item()
        train_forecast_mae_sum += forecast_mae * batch_size

        # 4. Forcast MSE
        forecast_mse = torch.nn.functional.mse_loss(pred_forecast, future_data).item()
        train_forecast_mse_sum += forecast_mse * batch_size

    avg_train_loss = total_train_loss / len(train_loader)
    avg_kss_mae = train_kss_mae_sum / total_samples
    avg_kss_acc = train_kss_acc_sum / total_samples # in decimal
    avg_forecast_mae = train_forecast_mae_sum / total_samples
    avg_forecast_rmse = math.sqrt(train_forecast_mse_sum / total_samples)

    # Save history
    history['train_loss'].append(avg_train_loss)
    history['train_kss_mae'].append(avg_kss_mae)
    history['train_kss_acc'].append(avg_kss_acc)
    history['train_forecast_mae'].append(avg_forecast_mae)
    history['train_forecast_rmse'].append(avg_forecast_rmse)

    # VALIDATION
    model.eval()
    total_val_loss = 0.0
    val_kss_mae_sum = 0.0
    val_kss_acc_sum = 0.0
    val_forecast_mae_sum = 0.0
    val_forecast_mse_sum = 0.0

    total_samples = 0

    with torch.no_grad() :
        for data, label, future_data in val_loader:
            data = data.to(device)
            label = label.float().unsqueeze(1).to(device)
            future_data = future_data.to(device)

            batch_size = data.size(0)
            total_samples += batch_size
            
            pred_kss, pred_forecast = model(data)
            pred_forecast = pred_forecast.unsqueeze(2)

            rounded_kss = torch.round(pred_kss)
            rounded_kss = torch.clamp(rounded_kss, min=1.0, max=9.0)

            loss_kss = classification_criterion(rounded_kss, label)
            loss_forecast = forecasting_criterion(pred_forecast, future_data)
            
            batch_loss = (config.ALPHA * loss_kss) + ((1 - config.ALPHA) * loss_forecast)
            total_val_loss += batch_loss.item()

            # Calculate Metrics
            # 1. KSS MAE
            kss_mae = torch.abs(pred_kss - label).mean().item()
            val_kss_mae_sum += kss_mae * batch_size

            # 2. KSS Accuracy
            correct_kss = (rounded_kss == label).sum().item()
            val_kss_acc_sum += correct_kss

            # 3. Forecast MAE
            forecast_mae = torch.abs(pred_forecast - future_data).mean().item()
            val_forecast_mae_sum += forecast_mae * batch_size

            # 4. Forcast MSE
            forecast_mse = torch.nn.functional.mse_loss(pred_forecast, future_data).item()
            val_forecast_mse_sum += forecast_mse * batch_size
            
    avg_val_loss = total_val_loss / len(val_loader)
    avg_kss_mae = val_kss_mae_sum / total_samples
    avg_kss_acc = val_kss_acc_sum / total_samples # in decimal
    avg_forecast_mae = val_forecast_mae_sum / total_samples
    avg_forecast_rmse = math.sqrt(val_forecast_mse_sum / total_samples)

    # Save History
    history['val_loss'].append(avg_val_loss)
    history['val_kss_mae'].append(avg_kss_mae)
    history['val_kss_acc'].append(avg_kss_acc)
    history['val_forecast_mae'].append(avg_forecast_mae)
    history['val_forecast_rmse'].append(avg_forecast_rmse)
    print(f"Epoch [{epoch+1}/{config.EPOCH}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        # Save Model Weight
        f_model_path = f"model/weight_{timestamp}.safetensors"
        if not os.path.exists("model"):
            os.mkdir("model")
        torch.save(model.state_dict(), f_model_path)

    # Saving Checkpoint
    if not os.path.exists("log/checkpoint"):
        os.makedirs("log/checkpoint")
    f_checkpoint_path = f"log/checkpoint/checkpoint_{timestamp}.pth.tar"
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_val_loss': best_val_loss,
    }

    torch.save(checkpoint, f_checkpoint_path)
        
if not os.path.exists("log") :
    os.mkdir("log")
metrics = pd.DataFrame(history)
metrics.to_csv(f"log/trai_eval_{timestamp}.csv", index=False)

# Training from Pre-trained

In [ ]:
# from transformers import PatchTSTConfig, PatchTSTForClassification, PatchTSTFMForPrediction

# model_id = "ibm-granite/granite-timeseries-patchtst-fm-r1"
# config = PatchTSTConfig.from_pretrained(model_id)
# config.num_input_channels = 1
# config.context_length = 300
# config.num_labels = 2

# model = PatchTSTFMForPrediction.from_pretrained(
#     model_id, 
#     config=config, 
#     ignore_mismatched_sizes=True # Abaikan perbedaan ukuran output karena kita ubah num_labels-nya
# )

In [ ]:
# from transformers import TrainingArguments, Trainer

# training_args = TrainingArguments(
#     output_dir="./hasil_patchtst_vitalsign",
#     per_device_train_batch_size=32,
#     per_device_eval_batch_size=32,
#     num_train_epochs=5,
#     learning_rate=5e-5,
#     logging_steps=10,
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     fp16=torch.cuda.is_available(),
#     report_to="none"
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset
# )


# print("Starting fine-tuning PatchTST...")
# trainer.train()

In [ ]:
# test_dataset = HuggingFacePatchTSTDataset(df_test)

# predictions = trainer.predict(test_dataset)
# predicted_labels = np.argmax(predictions.predictions, axis=-1)

# print("Hasil prediksi model untuk 10 pasien pertama:", predicted_labels[:10])
# print("Kondisi asli pasien lapangan                  :", [test_dataset[i]['labels'].item() for i in range(10)])